<a href="https://colab.research.google.com/github/renzo1001/scraper-onpe-2026-peru/blob/main/ONPE_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# ONPE 2026 - SCRAPER ASYNC PARA GOOGLE COLAB
# Perú por provincia/distrito + extranjero por continente
# Versión corregida para evitar "HTML en vez de JSON"
# ============================================================

!pip install curl_cffi -q

import asyncio
import json
import csv
import io
import time
import zipfile
from datetime import datetime
from pathlib import Path

from curl_cffi.requests import AsyncSession

# ============================================================
# CONFIGURACIÓN
# ============================================================

BASE = "https://resultadoelectoral.onpe.gob.pe/presentacion-backend"
MAIN_URL = "https://resultadoelectoral.onpe.gob.pe/main/presidenciales"
MAIN_URL_ALT = "https://resultadoelectoral.onpe.gob.pe/main/resumen"

ID_ELECCION = 10

# Ejemplo: "140000" para solo Lima. None = todo Perú
SOLO_DEP = None

INCLUIR_EXT = True

# Si ONPE devuelve muchos errores, baja a 4.
# Si va estable, puedes subir a 8 o 10.
CONCURRENCY = 6

REINTENTOS = 4
TIMEOUT_SEG = 45

OUTPUT_DIR = Path("onpe_data")
DEBUG_DIR = OUTPUT_DIR / "debug"
OUTPUT_DIR.mkdir(exist_ok=True)
DEBUG_DIR.mkdir(exist_ok=True)

CONTINENTES = [
    {"nombre": "ÁFRICA", "ubigeo": "910000"},
    {"nombre": "AMÉRICA", "ubigeo": "920000"},
    {"nombre": "ASIA", "ubigeo": "930000"},
    {"nombre": "EUROPA", "ubigeo": "940000"},
    {"nombre": "OCEANÍA", "ubigeo": "950000"},
]

HEADERS = {
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "es-ES,es;q=0.9,en;q=0.8",
    "Referer": MAIN_URL,
    "Origin": "https://resultadoelectoral.onpe.gob.pe",
    "User-Agent": (
        "Mozilla/5.0 (X11; Linux x86_64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
}

HTML_HEADERS = {
    **HEADERS,
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}

# ============================================================
# HELPERS
# ============================================================

def to_csv(rows):
    if not rows:
        return ""
    keys = list(dict.fromkeys(k for r in rows for k in r))
    buf = io.StringIO()
    w = csv.DictWriter(buf, fieldnames=keys, extrasaction="ignore", lineterminator="\n")
    w.writeheader()
    w.writerows(rows)
    return buf.getvalue()

def get_list(payload):
    if isinstance(payload, dict) and isinstance(payload.get("data"), list):
        return payload["data"]
    if isinstance(payload, list):
        return payload
    return []

def get_dict(payload):
    if isinstance(payload, dict):
        d = payload.get("data")
        if isinstance(d, dict):
            return d
        if "actasContabilizadas" in payload:
            return payload
    return None

def pick_name(obj, fallback="?"):
    if not isinstance(obj, dict):
        return fallback
    for k in ("nombre", "descripcion", "label", "text"):
        if obj.get(k):
            return obj[k]
    return fallback

def pick_code(obj, keys):
    if not isinstance(obj, dict):
        return None
    for k in keys:
        v = obj.get(k)
        if v is not None and str(v) != "":
            return str(v)
    return None

def safe_label(text):
    return "".join(
        c if c.isalnum() or c in ("_", "-", ".") else "_"
        for c in str(text)
    )[:180]

def save_debug(label, intento, reason, text):
    path = DEBUG_DIR / f"{safe_label(label)}_try{intento}_{reason}.txt"
    try:
        path.write_text((text or "")[:500000], encoding="utf-8", errors="ignore")
    except Exception:
        pass
    return str(path)

async def warmup_session(session):
    """
    Abre primero páginas principales para que ONPE entregue cookies/sesión.
    """
    for url in [MAIN_URL, MAIN_URL_ALT]:
        try:
            await session.get(url, headers=HTML_HEADERS, timeout=TIMEOUT_SEG)
            await asyncio.sleep(1.0)
        except Exception as e:
            print(f"Aviso: no se pudo abrir {url}: {e}")

# ============================================================
# FETCH CON REINTENTOS
# ============================================================

async def fetch_json(session, url, label, sem):
    async with sem:
        for intento in range(1, REINTENTOS + 1):
            try:
                resp = await session.get(
                    url,
                    headers=HEADERS,
                    timeout=TIMEOUT_SEG,
                )

                text = resp.text or ""
                t = text.strip()
                status = getattr(resp, "status_code", None)
                content_type = resp.headers.get("content-type", "")

                # Caso típico: ONPE devuelve HTML del frontend en vez de JSON.
                if (
                    "text/html" in content_type.lower()
                    or t.lower().startswith("<!doctype")
                    or t.lower().startswith("<html")
                ):
                    debug_path = save_debug(label, intento, "html", text)

                    if intento < REINTENTOS:
                        try:
                            await warmup_session(session)
                        except Exception:
                            pass
                        await asyncio.sleep(1.5 * intento)
                        continue

                    return {
                        "ok": False,
                        "error": f"HTML en vez de JSON [status {status}] | debug: {debug_path}",
                        "url": url,
                        "label": label,
                    }

                try:
                    data = json.loads(text)
                    return {
                        "ok": True,
                        "data": data,
                        "url": url,
                        "label": label,
                    }
                except Exception as e:
                    debug_path = save_debug(label, intento, "no_json", text)

                    if intento < REINTENTOS:
                        await asyncio.sleep(1.5 * intento)
                        continue

                    return {
                        "ok": False,
                        "error": f"No parsea JSON [status {status}] | {e} | debug: {debug_path}",
                        "url": url,
                        "label": label,
                    }

            except Exception as e:
                if intento < REINTENTOS:
                    await asyncio.sleep(1.5 * intento)
                else:
                    return {
                        "ok": False,
                        "error": str(e),
                        "url": url,
                        "label": label,
                    }

# ============================================================
# CONSTRUCTORES DE URL
# ============================================================

def url_deps():
    return f"{BASE}/ubigeos/departamentos?idEleccion={ID_ELECCION}&idAmbitoGeografico=1"

def url_provs(dep):
    return (
        f"{BASE}/ubigeos/provincias"
        f"?idEleccion={ID_ELECCION}"
        f"&idAmbitoGeografico=1"
        f"&idUbigeoDepartamento={dep}"
    )

def url_dists(dep, prov):
    return (
        f"{BASE}/ubigeos/distritos"
        f"?idEleccion={ID_ELECCION}"
        f"&idAmbitoGeografico=1"
        f"&idUbigeoDepartamento={dep}"
        f"&idUbigeoProvincia={prov}"
    )

def url_part_prov(dep, prov):
    return (
        f"{BASE}/resumen-general/participantes"
        f"?idEleccion={ID_ELECCION}"
        f"&tipoFiltro=ubigeo_nivel_02"
        f"&idAmbitoGeografico=1"
        f"&idUbigeoDepartamento={dep}"
        f"&idUbigeoProvincia={prov}"
    )

def url_tot_prov(dep, prov):
    return (
        f"{BASE}/resumen-general/totales"
        f"?idEleccion={ID_ELECCION}"
        f"&idAmbitoGeografico=1"
        f"&tipoFiltro=ubigeo_nivel_02"
        f"&idUbigeoDepartamento={dep}"
        f"&idUbigeoProvincia={prov}"
    )

def url_part_dist(dep, prov, dist):
    return (
        f"{BASE}/resumen-general/participantes"
        f"?idEleccion={ID_ELECCION}"
        f"&tipoFiltro=ubigeo_nivel_03"
        f"&idAmbitoGeografico=1"
        f"&idUbigeoDepartamento={dep}"
        f"&idUbigeoProvincia={prov}"
        f"&idUbigeoDistrito={dist}"
    )

def url_tot_dist(dep, prov, dist):
    return (
        f"{BASE}/resumen-general/totales"
        f"?idEleccion={ID_ELECCION}"
        f"&idAmbitoGeografico=1"
        f"&tipoFiltro=ubigeo_nivel_03"
        f"&idUbigeoDepartamento={dep}"
        f"&idUbigeoProvincia={prov}"
        f"&idUbigeoDistrito={dist}"
    )

def url_part_ext(ubigeo):
    return (
        f"{BASE}/eleccion-presidencial/participantes-ubicacion-geografica-nombre"
        f"?tipoFiltro=ubigeo_nivel_01"
        f"&idAmbitoGeografico=2"
        f"&ubigeoNivel1={ubigeo}"
        f"&listDepartamento="
        f"&listProvincia="
        f"&listDistrito="
        f"&idEleccion={ID_ELECCION}"
    )

def url_tot_ext(ubigeo):
    return (
        f"{BASE}/resumen-general/totales"
        f"?idAmbitoGeografico=2"
        f"&idEleccion={ID_ELECCION}"
        f"&tipoFiltro=ubigeo_nivel_01"
        f"&idUbigeoDepartamento={ubigeo}"
    )

# ============================================================
# PARSERS
# ============================================================

def parse_participantes(rows, ambito, nivel, dep_n, dep_c, prov_n, prov_c, dist_n="", dist_c=""):
    out = []
    for f in rows:
        out.append({
            "ambito": ambito,
            "nivel": nivel,
            "departamento": dep_n,
            "idUbigeoDepartamento": dep_c,
            "provincia": prov_n,
            "idUbigeoProvincia": prov_c,
            "distrito": dist_n,
            "idUbigeoDistrito": dist_c,
            "nombreAgrupacionPolitica": f.get("nombreAgrupacionPolitica", ""),
            "codigoAgrupacionPolitica": f.get("codigoAgrupacionPolitica", ""),
            "nombreCandidato": f.get("nombreCandidato", ""),
            "dniCandidato": f.get("dniCandidato", ""),
            "totalVotosValidos": f.get("totalVotosValidos", ""),
            "porcentajeVotosValidos": f.get("porcentajeVotosValidos", ""),
            "porcentajeVotosEmitidos": f.get("porcentajeVotosEmitidos", ""),
        })
    return out

def parse_totales(t, ambito, nivel, dep_n, dep_c, prov_n, prov_c, dist_n="", dist_c=""):
    if not t:
        return None

    total_actas = t.get("totalActas") or 0
    contabilizadas = t.get("contabilizadas") or 0

    return {
        "ambito": ambito,
        "nivel": nivel,
        "departamento": dep_n,
        "idUbigeoDepartamento": dep_c,
        "provincia": prov_n,
        "idUbigeoProvincia": prov_c,
        "distrito": dist_n,
        "idUbigeoDistrito": dist_c,
        "actasContabilizadas_pct": t.get("actasContabilizadas", ""),
        "contabilizadas": t.get("contabilizadas", ""),
        "totalActas": t.get("totalActas", ""),
        "actasPendientesCalc": total_actas - contabilizadas,
        "participacionCiudadana": t.get("participacionCiudadana", ""),
        "actasEnviadasJee_pct": t.get("actasEnviadasJee", ""),
        "enviadasJee": t.get("enviadasJee", ""),
        "actasPendientesJee_pct": t.get("actasPendientesJee", ""),
        "pendientesJee": t.get("pendientesJee", ""),
        "fechaActualizacion": t.get("fechaActualizacion", ""),
        "totalVotosEmitidos": t.get("totalVotosEmitidos", ""),
        "totalVotosValidos": t.get("totalVotosValidos", ""),
        "porcentajeVotosEmitidos": t.get("porcentajeVotosEmitidos", ""),
        "porcentajeVotosValidos": t.get("porcentajeVotosValidos", ""),
    }

# ============================================================
# MAIN
# ============================================================

async def main():
    sem = asyncio.Semaphore(CONCURRENCY)

    provincias_catalogo = []
    distritos_catalogo = []
    participantes_provincia = []
    avance_provincia = []
    participantes_distrito = []
    avance_distrito = []
    participantes_extranjero = []
    avance_extranjero = []
    errores = []

    async with AsyncSession(
        impersonate="chrome124",
        headers=HEADERS,
        timeout=TIMEOUT_SEG,
    ) as session:

        print("Abriendo página principal ONPE para iniciar sesión...")
        await warmup_session(session)

        # ----------------------------------------------------
        # 1) DEPARTAMENTOS
        # ----------------------------------------------------
        print("\nDescargando departamentos...")
        dep_res = await fetch_json(session, url_deps(), "departamentos", sem)

        if not dep_res["ok"]:
            print("ERROR al descargar departamentos:", dep_res["error"])
            print("Revisa la carpeta:", DEBUG_DIR.absolute())
            return

        departamentos = get_list(dep_res["data"])

        if SOLO_DEP:
            departamentos = [
                d for d in departamentos
                if pick_code(d, ["ubigeo", "idUbigeoDepartamento", "codigo"]) == SOLO_DEP
            ]

        print(f"Departamentos a procesar: {len(departamentos)}")
        t0 = time.time()

        # ----------------------------------------------------
        # 2) PERÚ
        # ----------------------------------------------------
        for i, dep in enumerate(departamentos):
            dep_n = pick_name(dep, f"DEP_{i + 1}")
            dep_c = pick_code(dep, ["idUbigeoDepartamento", "ubigeo", "codigo"])

            print(f"\n[{i + 1}/{len(departamentos)}] Departamento: {dep_n} ({dep_c})")

            # Provincias
            prov_res = await fetch_json(session, url_provs(dep_c), f"provs_{dep_c}", sem)
            if not prov_res["ok"]:
                errores.append({
                    "tipo": "catalogo_provincias",
                    "departamento": dep_n,
                    "idUbigeoDepartamento": dep_c,
                    "error": prov_res["error"],
                    "url": prov_res["url"],
                })
                print(f"  ERROR provincias: {prov_res['error']}")
                continue

            provincias = get_list(prov_res["data"])

            for p in provincias:
                prov_n = pick_name(p, "?")
                prov_c = pick_code(p, ["idUbigeoProvincia", "ubigeo", "codigo", "idUbigeo"])
                provincias_catalogo.append({
                    "departamento": dep_n,
                    "idUbigeoDepartamento": dep_c,
                    "provincia": prov_n,
                    "idUbigeoProvincia": prov_c,
                })

            print(f"  Provincias encontradas: {len(provincias)}")

            # Nivel provincia en paralelo
            prov_tasks = []
            for idx, p in enumerate(provincias):
                prov_c = pick_code(p, ["idUbigeoProvincia", "ubigeo", "codigo", "idUbigeo"])
                prov_tasks.append(
                    asyncio.gather(
                        fetch_json(session, url_part_prov(dep_c, prov_c), f"part_prov_{dep_c}_{prov_c}", sem),
                        fetch_json(session, url_tot_prov(dep_c, prov_c), f"tot_prov_{dep_c}_{prov_c}", sem),
                    )
                )

            prov_results = await asyncio.gather(*prov_tasks) if prov_tasks else []

            for idx, p in enumerate(provincias):
                prov_n = pick_name(p, "?")
                prov_c = pick_code(p, ["idUbigeoProvincia", "ubigeo", "codigo", "idUbigeo"])
                part_r, tot_r = prov_results[idx]

                if part_r["ok"]:
                    participantes_provincia.extend(
                        parse_participantes(
                            get_list(part_r["data"]),
                            "PERÚ",
                            "provincia",
                            dep_n,
                            dep_c,
                            prov_n,
                            prov_c,
                        )
                    )
                else:
                    errores.append({
                        "tipo": "participantes_provincia",
                        "departamento": dep_n,
                        "provincia": prov_n,
                        "error": part_r["error"],
                        "url": part_r["url"],
                    })

                if tot_r["ok"]:
                    r = parse_totales(
                        get_dict(tot_r["data"]),
                        "PERÚ",
                        "provincia",
                        dep_n,
                        dep_c,
                        prov_n,
                        prov_c,
                    )
                    if r:
                        avance_provincia.append(r)
                else:
                    errores.append({
                        "tipo": "totales_provincia",
                        "departamento": dep_n,
                        "provincia": prov_n,
                        "error": tot_r["error"],
                        "url": tot_r["url"],
                    })

            # Nivel distrito
            for p in provincias:
                prov_n = pick_name(p, "?")
                prov_c = pick_code(p, ["idUbigeoProvincia", "ubigeo", "codigo", "idUbigeo"])

                dist_res = await fetch_json(session, url_dists(dep_c, prov_c), f"dists_{dep_c}_{prov_c}", sem)
                if not dist_res["ok"]:
                    errores.append({
                        "tipo": "catalogo_distritos",
                        "departamento": dep_n,
                        "provincia": prov_n,
                        "error": dist_res["error"],
                        "url": dist_res["url"],
                    })
                    print(f"  ERROR distritos en {prov_n}: {dist_res['error']}")
                    continue

                distritos = get_list(dist_res["data"])

                dist_list = []
                for d in distritos:
                    dist_n = pick_name(d, "?")
                    dist_c = pick_code(d, ["idUbigeoDistrito", "ubigeo", "codigo", "idUbigeo"])
                    dist_list.append((dist_n, dist_c))

                    distritos_catalogo.append({
                        "departamento": dep_n,
                        "idUbigeoDepartamento": dep_c,
                        "provincia": prov_n,
                        "idUbigeoProvincia": prov_c,
                        "distrito": dist_n,
                        "idUbigeoDistrito": dist_c,
                    })

                print(f"  {dep_n} / {prov_n}: {len(dist_list)} distritos")

                # Procesar distritos en lotes
                batch_size = CONCURRENCY
                for b in range(0, len(dist_list), batch_size):
                    lote = dist_list[b:b + batch_size]

                    tasks = [
                        asyncio.gather(
                            fetch_json(session, url_part_dist(dep_c, prov_c, dc), f"part_dist_{dc}", sem),
                            fetch_json(session, url_tot_dist(dep_c, prov_c, dc), f"tot_dist_{dc}", sem),
                        )
                        for dn, dc in lote
                    ]

                    results = await asyncio.gather(*tasks) if tasks else []

                    for idx, (dn, dc) in enumerate(lote):
                        part_r, tot_r = results[idx]

                        if part_r["ok"]:
                            participantes_distrito.extend(
                                parse_participantes(
                                    get_list(part_r["data"]),
                                    "PERÚ",
                                    "distrito",
                                    dep_n,
                                    dep_c,
                                    prov_n,
                                    prov_c,
                                    dn,
                                    dc,
                                )
                            )
                        else:
                            errores.append({
                                "tipo": "participantes_distrito",
                                "departamento": dep_n,
                                "provincia": prov_n,
                                "distrito": dn,
                                "error": part_r["error"],
                                "url": part_r["url"],
                            })

                        if tot_r["ok"]:
                            r = parse_totales(
                                get_dict(tot_r["data"]),
                                "PERÚ",
                                "distrito",
                                dep_n,
                                dep_c,
                                prov_n,
                                prov_c,
                                dn,
                                dc,
                            )
                            if r:
                                avance_distrito.append(r)
                        else:
                            errores.append({
                                "tipo": "totales_distrito",
                                "departamento": dep_n,
                                "provincia": prov_n,
                                "distrito": dn,
                                "error": tot_r["error"],
                                "url": tot_r["url"],
                            })

            elapsed = time.time() - t0
            eta = elapsed / (i + 1) * (len(departamentos) - i - 1) if (i + 1) > 0 else 0
            prov_count = sum(1 for x in provincias_catalogo if x["idUbigeoDepartamento"] == dep_c)
            dist_count = sum(1 for x in distritos_catalogo if x["idUbigeoDepartamento"] == dep_c)

            print(
                f"  OK {dep_n}: {prov_count} provincias, {dist_count} distritos | "
                f"{elapsed:.0f}s transcurridos, ETA {eta:.0f}s"
            )

        # ----------------------------------------------------
        # 3) EXTRANJERO
        # ----------------------------------------------------
        if INCLUIR_EXT:
            print("\nProcesando extranjero por continente...")

            ext_tasks = [
                asyncio.gather(
                    fetch_json(session, url_part_ext(c["ubigeo"]), f"part_ext_{c['ubigeo']}", sem),
                    fetch_json(session, url_tot_ext(c["ubigeo"]), f"tot_ext_{c['ubigeo']}", sem),
                )
                for c in CONTINENTES
            ]

            ext_results = await asyncio.gather(*ext_tasks)

            for idx, c in enumerate(CONTINENTES):
                cn = c["nombre"]
                cu = c["ubigeo"]
                part_r, tot_r = ext_results[idx]

                if part_r["ok"]:
                    for f in get_list(part_r["data"]):
                        participantes_extranjero.append({
                            "ambito": "EXTRANJERO",
                            "nivel": "continente",
                            "continente": cn,
                            "ubigeo": cu,
                            "nombreAgrupacionPolitica": f.get("nombreAgrupacionPolitica", ""),
                            "codigoAgrupacionPolitica": f.get("codigoAgrupacionPolitica", ""),
                            "nombreCandidato": f.get("nombreCandidato", ""),
                            "dniCandidato": f.get("dniCandidato", ""),
                            "totalVotosValidos": f.get("totalVotosValidos", ""),
                            "porcentajeVotosValidos": f.get("porcentajeVotosValidos", ""),
                            "porcentajeVotosEmitidos": f.get("porcentajeVotosEmitidos", ""),
                        })
                else:
                    errores.append({
                        "tipo": "participantes_extranjero",
                        "continente": cn,
                        "error": part_r["error"],
                        "url": part_r["url"],
                    })

                if tot_r["ok"]:
                    t = get_dict(tot_r["data"])
                    if t:
                        avance_extranjero.append({
                            "ambito": "EXTRANJERO",
                            "nivel": "continente",
                            "continente": cn,
                            "ubigeo": cu,
                            "actasContabilizadas_pct": t.get("actasContabilizadas", ""),
                            "contabilizadas": t.get("contabilizadas", ""),
                            "totalActas": t.get("totalActas", ""),
                            "actasPendientesCalc": (t.get("totalActas") or 0) - (t.get("contabilizadas") or 0),
                            "participacionCiudadana": t.get("participacionCiudadana", ""),
                            "actasEnviadasJee_pct": t.get("actasEnviadasJee", ""),
                            "enviadasJee": t.get("enviadasJee", ""),
                            "actasPendientesJee_pct": t.get("actasPendientesJee", ""),
                            "pendientesJee": t.get("pendientesJee", ""),
                            "fechaActualizacion": t.get("fechaActualizacion", ""),
                            "totalVotosEmitidos": t.get("totalVotosEmitidos", ""),
                            "totalVotosValidos": t.get("totalVotosValidos", ""),
                            "porcentajeVotosEmitidos": t.get("porcentajeVotosEmitidos", ""),
                            "porcentajeVotosValidos": t.get("porcentajeVotosValidos", ""),
                        })
                else:
                    errores.append({
                        "tipo": "totales_extranjero",
                        "continente": cn,
                        "error": tot_r["error"],
                        "url": tot_r["url"],
                    })

    # --------------------------------------------------------
    # 4) GUARDAR ARCHIVOS
    # --------------------------------------------------------
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")

    paquete = {
        "generado_en": datetime.now().isoformat(),
        "idEleccion": ID_ELECCION,
        "config": {
            "SOLO_DEP": SOLO_DEP,
            "INCLUIR_EXT": INCLUIR_EXT,
            "CONCURRENCY": CONCURRENCY,
            "REINTENTOS": REINTENTOS,
        },
        "ambitos": {
            "peru": {
                "provincias_catalogo": provincias_catalogo,
                "distritos_catalogo": distritos_catalogo,
                "participantes_provincia": participantes_provincia,
                "avance_provincia": avance_provincia,
                "participantes_distrito": participantes_distrito,
                "avance_distrito": avance_distrito,
            },
            "extranjero": {
                "participantes_extranjero": participantes_extranjero,
                "avance_extranjero": avance_extranjero,
            },
        },
        "errores": errores,
    }

    archivos = {
        f"onpe_completo_distrito_y_extranjero_{ts}.json": json.dumps(paquete, ensure_ascii=False, indent=2),
        f"onpe_participantes_provincia_{ts}.csv": to_csv(participantes_provincia),
        f"onpe_avance_provincia_{ts}.csv": to_csv(avance_provincia),
        f"onpe_participantes_distrito_{ts}.csv": to_csv(participantes_distrito),
        f"onpe_avance_distrito_{ts}.csv": to_csv(avance_distrito),
        f"onpe_participantes_extranjero_{ts}.csv": to_csv(participantes_extranjero),
        f"onpe_avance_extranjero_{ts}.csv": to_csv(avance_extranjero),
        f"onpe_errores_{ts}.csv": to_csv(errores),
    }

    saved_paths = []

    for fname, content in archivos.items():
        path = OUTPUT_DIR / fname
        path.write_text(content, encoding="utf-8")
        saved_paths.append(path)

    # ZIP para descargar todo junto
    zip_path = OUTPUT_DIR / f"onpe_data_{ts}.zip"
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
        for path in saved_paths:
            z.write(path, arcname=path.name)

        # incluir debug si existe
        if DEBUG_DIR.exists():
            for dbg in DEBUG_DIR.glob("*.txt"):
                z.write(dbg, arcname=f"debug/{dbg.name}")

    total = time.time() - t0 if "t0" in locals() else 0

    print("\n================ RESUMEN ================")
    print(f"Completado en: {total/60:.1f} minutos")
    print(f"Provincias catálogo:      {len(provincias_catalogo)}")
    print(f"Distritos catálogo:       {len(distritos_catalogo)}")
    print(f"Participantes provincia:  {len(participantes_provincia)}")
    print(f"Avance provincia:         {len(avance_provincia)}")
    print(f"Participantes distrito:   {len(participantes_distrito)}")
    print(f"Avance distrito:          {len(avance_distrito)}")
    print(f"Participantes extranjero: {len(participantes_extranjero)}")
    print(f"Avance extranjero:        {len(avance_extranjero)}")
    print(f"Errores:                  {len(errores)}")
    print(f"Carpeta:                  {OUTPUT_DIR.absolute()}")
    print(f"ZIP:                      {zip_path.absolute()}")

    if errores:
        print("\nPrimeros errores:")
        for e in errores[:10]:
            print(e)

    # Descargar ZIP automáticamente en Colab
    try:
        from google.colab import files
        files.download(str(zip_path))
    except Exception as e:
        print("No se pudo descargar automáticamente. Descarga manualmente desde:", zip_path)

# ============================================================
# EJECUTAR
# ============================================================

await main()